# Chapter 1 — Arrays & Strings

**DS/MLE perspective first:** You use `rolling()`, `cumsum()`, and `merge_asof()` every day. This chapter shows the algorithms underneath those tools — and the two LeetCode problems per pattern that test whether you can write those algorithms from scratch.

Three patterns, same order as `notes.md`:
1. **Two pointers** → `pd.merge_asof()`, sort-merge joins — LC 167, LC 125
2. **Sliding window** → `pd.rolling()`, time-series features — LC 643, LC 3
3. **Prefix sum** → `np.cumsum()`, fast range queries — LC 303, LC 560

Each section: DS/ML version first → mechanism → LeetCode problem → side-by-side implementations → step-by-step trace.

In [45]:
import numpy as np
import pandas as pd
import time
from collections import defaultdict, deque

# ── Trace helpers ──────────────────────────────────────────────────────────────

def show_array(arr, markers=None, width=5):
    """
    Print an array with optional labeled pointer markers below each index.
    markers: dict of {index: label}, e.g. {0: 'L', 3: 'R'}
    """
    row   = ''.join(f"{str(v):>{width}}" for v in arr)
    index = ''.join(f"{i:>{width}}" for i in range(len(arr)))
    print(f"  vals : {row}")
    print(f"  idx  : {index}")
    if markers:
        ptr = [''] * len(arr)
        for idx, label in markers.items():
            if 0 <= idx < len(arr):
                ptr[idx] = label
        print(f"         {''.join(f'{p:>{width}}' for p in ptr)}")

def show_window(arr, left, right, width=5):
    """Print array with brackets marking the current window [left..right]."""
    parts = []
    for i, v in enumerate(arr):
        s = f"{str(v):>{width}}"
        if i == left:  s = '[' + s.lstrip()
        if i == right: s = s.rstrip() + ']'
        parts.append(s)
    print('  ' + '  '.join(parts))

---
## Part 1 — Two Pointers

### What you already use: `pd.merge_asof()`

When you align two time series at different frequencies — matching sparse news events to the most recent stock price — pandas does it with a two-pointer merge on sorted timestamps.

In [46]:
np.random.seed(42)
prices = pd.DataFrame({
    'time':  sorted(np.random.randint(0, 100, 20)),
    'price': np.round(100 + np.random.randn(20), 2)
})
events = pd.DataFrame({
    'time':  sorted(np.random.randint(0, 100, 5)),
    'event': [f'event_{i}' for i in range(5)]
})

# merge_asof: for each event, find the most recent preceding price
# Under the hood: two pointers, one per DataFrame, advancing through sorted timestamps
aligned = pd.merge_asof(events, prices, on='time')
print("Events aligned to most recent price:")
print(aligned.to_string(index=False))

Events aligned to most recent price:
 time   event  price
    7 event_0 101.54
   43 event_1 101.49
   46 event_2 101.49
   59 event_3 100.36
   70 event_4 100.42


### Under the hood: the two-pointer mechanism

`merge_asof` maintains a pointer `i` into prices and a pointer `j` into events. For each event `j`, it advances `i` as far as possible without overshooting `events.time[j]`. One pass through both arrays: O(n + m).

The same logic drives the classic interview problems below — one or two pointers moving through a sorted sequence, advancing based on a simple condition.

---
### LC 167 — [Two Sum II: Input Array Is Sorted](https://leetcode.com/problems/two-sum-ii-input-array-is-sorted/)

> Given a **1-indexed** sorted array `numbers` and a `target`, return the indices `[i, j]` of the two numbers that sum to `target`. Exactly one solution; O(1) extra space.
>
> Example: `numbers = [2, 7, 11, 15], target = 9` → `[1, 2]`

**DS parallel:** Finding two prices/scores that sum to a budget in a sorted candidate set.

**Two-pointer insight:** Because the array is sorted, if the sum is too small you *must* advance `left` (no larger element can help from the right for this left). If too large, retreat `right`. Each pointer moves at most n steps → O(n).

In [47]:
# ── NUMPY APPROACH (what a DS would reach for first) ──────────────────────────
# Binary search for each element's complement. Works, but O(n log n).
def two_sum_numpy(numbers, target):
    arr = np.array(numbers)
    for i, n in enumerate(arr):
        complement = target - n
        idx = np.searchsorted(arr[i+1:], complement)
        if idx < len(arr) - i - 1 and arr[i + 1 + idx] == complement:
            return [i + 1, i + 2 + idx]  # 1-indexed
    return []

# ── TWO POINTERS (the interview answer) ───────────────────────────────────────
# O(n) time, O(1) space.
def twoSum(numbers, target):
    left, right = 0, len(numbers) - 1
    while left < right:
        s = numbers[left] + numbers[right]
        if s == target:
            return [left + 1, right + 1]   # 1-indexed
        elif s < target:
            left += 1    # sum too small: need a larger value
        else:
            right -= 1   # sum too large: need a smaller value
    return []

cases = [
    ([2, 7, 11, 15], 9,  [1, 2]),
    ([2, 3, 4],      6,  [1, 3]),
    ([-1, 0],       -1,  [1, 2]),
]
print(f"{'numbers':<20} {'target':>6}  {'numpy':>8}  {'two-ptr':>8}  {'expected':>8}")
print("-" * 60)
for nums, tgt, expected in cases:
    np_ans = two_sum_numpy(nums, tgt)
    tp_ans = twoSum(nums, tgt)
    match = "✓" if tp_ans == expected else "✗"
    print(f"{str(nums):<20} {tgt:>6}  {str(np_ans):>8}  {str(tp_ans):>8}  {str(expected):>8}  {match}")

numbers              target     numpy   two-ptr  expected
------------------------------------------------------------
[2, 7, 11, 15]            9  [1, np.int64(2)]    [1, 2]    [1, 2]  ✓
[2, 3, 4]                 6  [1, np.int64(3)]    [1, 3]    [1, 3]  ✓
[-1, 0]                  -1  [1, np.int64(2)]    [1, 2]    [1, 2]  ✓


In [48]:
# ── Step-by-step trace with pointer visualization ─────────────────────────────
nums, target = [2, 7, 11, 15], 9
print(f"numbers = {nums},  target = {target}")
print()

left, right = 0, len(nums) - 1
step = 1
while left < right:
    s = nums[left] + nums[right]
    print(f"Step {step}:")
    show_array(nums, markers={left: 'L', right: 'R'})
    if s == target:
        print(f"  nums[L]={nums[left]} + nums[R]={nums[right]} = {s} == target  →  found! return [{left+1}, {right+1}]")
        break
    elif s < target:
        print(f"  nums[L]={nums[left]} + nums[R]={nums[right]} = {s} < {target}  →  sum too small, left++")
        left += 1
    else:
        print(f"  nums[L]={nums[left]} + nums[R]={nums[right]} = {s} > {target}  →  sum too large, right--")
        right -= 1
    print()
    step += 1

numbers = [2, 7, 11, 15],  target = 9

Step 1:
  vals :     2    7   11   15
  idx  :     0    1    2    3
             L              R
  nums[L]=2 + nums[R]=15 = 17 > 9  →  sum too large, right--

Step 2:
  vals :     2    7   11   15
  idx  :     0    1    2    3
             L         R     
  nums[L]=2 + nums[R]=11 = 13 > 9  →  sum too large, right--

Step 3:
  vals :     2    7   11   15
  idx  :     0    1    2    3
             L    R          
  nums[L]=2 + nums[R]=7 = 9 == target  →  found! return [1, 2]


---
### LC 125 — [Valid Palindrome](https://leetcode.com/problems/valid-palindrome/)

> After lowercasing and removing non-alphanumeric characters, does the string read the same forwards and backwards?
>
> Example: `"A man, a plan, a canal: Panama"` → `True`

**DS parallel:** Validating symmetric text fields without materializing the reversed copy.

**Two-pointer insight:** Start at opposite ends. Skip non-alphanumeric characters in place — no cleaned string built. Compare and move inward. O(n) time, **O(1) space**.

In [49]:
# ── PYTHONIC ONELINER (readable but O(n) space — creates a reversed copy) ─────
def isPalindrome_pythonic(s):
    cleaned = ''.join(c.lower() for c in s if c.isalnum())
    return cleaned == cleaned[::-1]

# ── TWO POINTERS (O(1) space — the interview answer) ──────────────────────────
def isPalindrome(s):
    left, right = 0, len(s) - 1
    while left < right:
        while left < right and not s[left].isalnum():  left += 1
        while left < right and not s[right].isalnum(): right -= 1
        if s[left].lower() != s[right].lower():
            return False
        left += 1
        right -= 1
    return True

cases = [
    ("A man, a plan, a canal: Panama", True),
    ("race a car",                     False),
    (" ",                              True),
    ("Was it a car or a cat I saw?",   True),
]
print(f"{'string':<35} {'pythonic':>10}  {'two-ptr':>10}  {'expected':>10}")
print("-" * 70)
for s, expected in cases:
    p = isPalindrome_pythonic(s)
    t = isPalindrome(s)
    match = "✓" if p == t == expected else "✗"
    print(f"{s!r:<35} {str(p):>10}  {str(t):>10}  {str(expected):>10}  {match}")

string                                pythonic     two-ptr    expected
----------------------------------------------------------------------
'A man, a plan, a canal: Panama'          True        True        True  ✓
'race a car'                             False       False       False  ✓
' '                                       True        True        True  ✓
'Was it a car or a cat I saw?'            True        True        True  ✓


In [50]:
# ── Step-by-step trace — shows pointer positions and skip behavior ─────────────
s = "A man, a plan"
print(f"Input: {s!r}")
print(f"Cleaned would be: {''.join(c.lower() for c in s if c.isalnum())!r}")
print()

# Print the character map for reference
print("Character map:")
print("  idx : " + ''.join(f"{i:>3}" for i in range(len(s))))
print("  chr : " + ''.join(f"{c:>3}" for c in s))
print("  alph: " + ''.join(f"{'Y' if c.isalnum() else '-':>3}" for c in s))
print()

left, right = 0, len(s) - 1
step = 1
while left < right:
    # Track which indices get skipped
    skipped_left, skipped_right = [], []
    while left < right and not s[left].isalnum():
        skipped_left.append(f"{left}({s[left]!r})"); left += 1
    while left < right and not s[right].isalnum():
        skipped_right.append(f"{right}({s[right]!r})"); right -= 1

    if left >= right: break

    lc, rc = s[left].lower(), s[right].lower()
    match = lc == rc

    print(f"Step {step}:")
    # Visual: show string with L/R markers
    markers = list(' ' * len(s))
    markers[left]  = 'L'
    markers[right] = 'R'
    print(f"  str : {s}")
    print(f"  ptrs: {''.join(markers)}")
    if skipped_left:  print(f"  skipped left  → {', '.join(skipped_left)} (non-alphanumeric)")
    if skipped_right: print(f"  skipped right → {', '.join(skipped_right)} (non-alphanumeric)")
    cmp    = "==" if match else "!="
    result = "match → move inward" if match else "MISMATCH → not a palindrome"
    print(f"  compare: s[{left}]={s[left]!r} {cmp} s[{right}]={s[right]!r}  →  {result}")
    print()

    if not match: break
    left += 1; right -= 1
    step += 1

print(f"Result: {isPalindrome(s)}")

Input: 'A man, a plan'
Cleaned would be: 'amanaplan'

Character map:
  idx :   0  1  2  3  4  5  6  7  8  9 10 11 12
  chr :   A     m  a  n  ,     a     p  l  a  n
  alph:   Y  -  Y  Y  Y  -  -  Y  -  Y  Y  Y  Y

Step 1:
  str : A man, a plan
  ptrs: L           R
  compare: s[0]='A' != s[12]='n'  →  MISMATCH → not a palindrome

Result: False


---
## Part 2 — Sliding Window

### What you already use: `pd.rolling()`

Rolling statistics are the bread and butter of time-series feature engineering. `pd.rolling(k)` is a fixed sliding window: right pointer always advances, left pointer lags by exactly `k`.

In [51]:
np.random.seed(42)
n = 100
latency = pd.Series(
    np.random.normal(50, 10, n) + np.sin(np.linspace(0, 4*np.pi, n)) * 15,
    name='latency_ms'
)

k = 10
df_display = pd.DataFrame({
    'latency':      latency,
    'rolling_mean': latency.rolling(k).mean(),
    'rolling_max':  latency.rolling(k).max()
})
print(f"Window size k={k}. First {k+3} rows (NaN until window fills):")
print(df_display.head(k + 3).round(1).to_string())

Window size k=10. First 13 rows (NaN until window fills):
    latency  rolling_mean  rolling_max
0      55.0           NaN          NaN
1      50.5           NaN          NaN
2      60.2           NaN          NaN
3      70.8           NaN          NaN
4      55.0           NaN          NaN
5      56.6           NaN          NaN
6      76.1           NaN          NaN
7      69.3           NaN          NaN
8      58.1           NaN          NaN
9      69.1          62.1         76.1
10     59.7          62.5         76.1
11     60.1          63.5         76.1
12     67.4          64.2         76.1


### Under the hood: fixed vs variable windows

**Fixed window (size k):** Maintain a running sum. Each step: add `arr[right]`, subtract `arr[right - k]`. One pass, O(n).

**Variable window:** `right` always advances. `left` only moves when the window violates a constraint. Because each element enters and leaves at most once, the total work is O(n) even with the inner `while`.

---
### LC 643 — [Maximum Average Subarray I](https://leetcode.com/problems/maximum-average-subarray-i/)

> Find the contiguous subarray of length exactly `k` with the maximum average. Return the average.
>
> Example: `nums = [1, 12, -5, -6, 50, 3], k = 4` → `12.75`

**DS parallel:** `pd.rolling(k).mean().max()` — find the window with the best average signal.

**Fixed window insight:** Don't recompute from scratch. Maintain a running sum: subtract the element leaving the left, add the one entering the right. One line changes per step.

In [52]:
# ── PANDAS (what you'd write in a DS notebook) ────────────────────────────────
def findMaxAverage_pandas(nums, k):
    return pd.Series(nums).rolling(k).mean().max()

# ── FIXED SLIDING WINDOW (the interview answer) ───────────────────────────────
# O(n) time, O(1) space.
def findMaxAverage(nums, k):
    window_sum = sum(nums[:k])
    best = window_sum
    for i in range(k, len(nums)):
        window_sum += nums[i] - nums[i - k]   # add right, drop left
        best = max(best, window_sum)
    return best / k

cases = [
    ([1, 12, -5, -6, 50, 3], 4, 12.75),
    ([5],                    1, 5.0),
    ([0, 1, 1, 3, 3],        4, 2.0),
]
print(f"{'nums':<28} {'k':>2}  {'pandas':>8}  {'sliding':>8}  {'expected':>8}")
print("-" * 60)
for nums, k, expected in cases:
    p = findMaxAverage_pandas(nums, k)
    s = findMaxAverage(nums, k)
    match = "✓" if abs(s - expected) < 1e-9 else "✗"
    print(f"{str(nums):<28} {k:>2}  {p:>8.2f}  {s:>8.2f}  {expected:>8.2f}  {match}")

nums                          k    pandas   sliding  expected
------------------------------------------------------------
[1, 12, -5, -6, 50, 3]        4     12.75     12.75     12.75  ✓
[5]                           1      5.00      5.00      5.00  ✓
[0, 1, 1, 3, 3]               4      2.00      2.00      2.00  ✓


In [53]:
# ── Step-by-step trace with window bracket visualization ──────────────────────
nums, k = [1, 12, -5, -6, 50, 3], 4
print(f"nums = {nums},  k = {k}")
print()

window_sum = sum(nums[:k])
best       = window_sum
best_start = 0

print(f"Initial window (indices 0–{k-1}):")
show_window(nums, 0, k - 1)
print(f"  sum = {' + '.join(str(v) for v in nums[:k])} = {window_sum},  avg = {window_sum/k:.2f},  best = {best/k:.2f}")
print()

for i in range(k, len(nums)):
    left    = i - k + 1
    removed = nums[i - k]
    added   = nums[i]
    window_sum += added - removed
    is_new_best = window_sum > best
    if is_new_best:
        best       = window_sum
        best_start = left

    print(f"Slide to indices {left}–{i}:")
    show_window(nums, left, i)
    print(f"  drop {removed:>4} (idx {i-k}),  add {added:>4} (idx {i})")
    print(f"  window_sum = {window_sum},  avg = {window_sum/k:.2f},  best = {best/k:.2f}" +
          ("  ← new best!" if is_new_best else ""))
    print()

print(f"Answer: {best}/{k} = {best/k:.2f}")
print(f"Best window: indices {best_start}–{best_start+k-1}  →  {nums[best_start:best_start+k]}")

nums = [1, 12, -5, -6, 50, 3],  k = 4

Initial window (indices 0–3):
  [1     12     -5     -6]     50      3
  sum = 1 + 12 + -5 + -6 = 2,  avg = 0.50,  best = 0.50

Slide to indices 1–4:
      1  [12     -5     -6     50]      3
  drop    1 (idx 0),  add   50 (idx 4)
  window_sum = 51,  avg = 12.75,  best = 12.75  ← new best!

Slide to indices 2–5:
      1     12  [-5     -6     50      3]
  drop   12 (idx 1),  add    3 (idx 5)
  window_sum = 42,  avg = 10.50,  best = 12.75

Answer: 51/4 = 12.75
Best window: indices 1–4  →  [12, -5, -6, 50]


---
### LC 3 — [Longest Substring Without Repeating Characters](https://leetcode.com/problems/longest-substring-without-repeating-characters/)

> Find the length of the longest substring with no repeating characters.
>
> Example: `s = "abcabcbb"` → `3`  (substring `"abc"`)

**DS parallel:** Longest window of log lines with no repeated error code; longest token span with no repeated n-gram. No `pd.rolling` handles this — window size isn't fixed.

**Variable window insight:** When a duplicate enters from the right, shrink from the left until it's gone. A freq map tracks counts. Each character enters and leaves at most once → O(n).

In [54]:
# ── NAIVE O(n²) ───────────────────────────────────────────────────────────────
def lengthOfLongestSubstring_naive(s):
    best = 0
    for i in range(len(s)):
        seen = set()
        for j in range(i, len(s)):
            if s[j] in seen: break
            seen.add(s[j])
            best = max(best, j - i + 1)
    return best

# ── VARIABLE SLIDING WINDOW (the interview answer) ────────────────────────────
# O(n) — each char enters the window once and leaves at most once.
def lengthOfLongestSubstring(s):
    freq = {}
    left = 0
    best = 0
    for right, char in enumerate(s):
        freq[char] = freq.get(char, 0) + 1
        while freq[char] > 1:          # duplicate in window — shrink from left
            freq[s[left]] -= 1
            if freq[s[left]] == 0: del freq[s[left]]
            left += 1
        best = max(best, right - left + 1)
    return best

cases = [
    ("abcabcbb", 3),
    ("bbbbb",    1),
    ("pwwkew",   3),
    ("",         0),
]
print(f"{'s':<12} {'naive':>7}  {'window':>7}  {'expected':>8}")
print("-" * 40)
for s, expected in cases:
    n_ans = lengthOfLongestSubstring_naive(s)
    w_ans = lengthOfLongestSubstring(s)
    match = "✓" if n_ans == w_ans == expected else "✗"
    print(f"{s!r:<12} {n_ans:>7}  {w_ans:>7}  {expected:>8}  {match}")

s              naive   window  expected
----------------------------------------
'abcabcbb'         3        3         3  ✓
'bbbbb'            1        1         1  ✓
'pwwkew'           3        3         3  ✓
''                 0        0         0  ✓


In [55]:
# ── Step-by-step trace — window content + freq map at every step ──────────────
s = "abcabcbb"
print(f"Input: {s!r}")
print()
print(f"  {'step':>4}  {'right':>5}  {'char':>5}  {'window':>10}  {'freq map':<28}  {'wlen':>4}  {'best':>4}  note")
print("  " + "-" * 80)

freq = {}
left = 0
best = 0
for step, (right, char) in enumerate(enumerate(s)):
    freq[char] = freq.get(char, 0) + 1

    shrinks = 0
    while freq[char] > 1:
        freq[s[left]] -= 1
        if freq[s[left]] == 0: del freq[s[left]]
        left += 1
        shrinks += 1

    prev_best = best
    best      = max(best, right - left + 1)

    window   = s[left:right+1]
    freq_str = "{" + ", ".join(f"{k}:{v}" for k, v in sorted(freq.items())) + "}"
    note     = ""
    if shrinks:        note += f"dup '{char}' → shrink {shrinks}×  "
    if best > prev_best: note += "new best!"

    print(f"  {step:>4}  {right:>5}  {char!r:>5}  {window!r:>10}  {freq_str:<28}  {right-left+1:>4}  {best:>4}  {note}")

print()
print(f"Answer: {best}")

Input: 'abcabcbb'

  step  right   char      window  freq map                      wlen  best  note
  --------------------------------------------------------------------------------
     0      0    'a'         'a'  {a:1}                            1     1  new best!
     1      1    'b'        'ab'  {a:1, b:1}                       2     2  new best!
     2      2    'c'       'abc'  {a:1, b:1, c:1}                  3     3  new best!
     3      3    'a'       'bca'  {a:1, b:1, c:1}                  3     3  dup 'a' → shrink 1×  
     4      4    'b'       'cab'  {a:1, b:1, c:1}                  3     3  dup 'b' → shrink 1×  
     5      5    'c'       'abc'  {a:1, b:1, c:1}                  3     3  dup 'c' → shrink 1×  
     6      6    'b'        'cb'  {b:1, c:1}                       2     3  dup 'b' → shrink 2×  
     7      7    'b'         'b'  {b:1}                            1     3  dup 'b' → shrink 2×  

Answer: 3


In [56]:
# DS application: longest span of requests with no repeated HTTP status code
print("DS application — longest span of requests with no repeated status code:")
status_codes = [200, 404, 200, 500, 200, 301, 404, 200]

freq2, left2, best2, best_window = {}, 0, 0, []
for right2, code in enumerate(status_codes):
    freq2[code] = freq2.get(code, 0) + 1
    while freq2[code] > 1:
        freq2[status_codes[left2]] -= 1
        if freq2[status_codes[left2]] == 0: del freq2[status_codes[left2]]
        left2 += 1
    if right2 - left2 + 1 > best2:
        best2       = right2 - left2 + 1
        best_window = status_codes[left2:right2+1]

print(f"  Status codes:          {status_codes}")
print(f"  Longest no-repeat window: {best_window} (length {best2})")

DS application — longest span of requests with no repeated status code:
  Status codes:          [200, 404, 200, 500, 200, 301, 404, 200]
  Longest no-repeat window: [500, 200, 301, 404] (length 4)


---
## Part 3 — Prefix Sum

### What you already use: `np.cumsum()`

Cumulative sums are everywhere in DS — cumulative revenue, cumulative returns, integral images. Build the prefix array once in O(n); answer any range sum query in O(1) with one subtraction.

In [57]:
np.random.seed(0)
daily_revenue = np.random.exponential(scale=1000, size=365)
prefix_np     = np.concatenate([[0], np.cumsum(daily_revenue)])

start, end  = 100, 200
range_total = prefix_np[end] - prefix_np[start]
print(f"Revenue days {start}–{end}: ${range_total:,.0f}")
print(f"Direct sum (same answer):   ${daily_revenue[start:end].sum():,.0f}")
print()
print("Why it matters — 10,000 random range queries:")
queries = [(np.random.randint(0, 300), np.random.randint(300, 365)) for _ in range(10_000)]

t0 = time.perf_counter()
for s, e in queries: _ = daily_revenue[s:e].sum()
naive_ms = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
for s, e in queries: _ = prefix_np[e] - prefix_np[s]
prefix_ms = (time.perf_counter() - t0) * 1000

print(f"  Naive  (O(n) per query): {naive_ms:.1f} ms")
print(f"  Prefix (O(1) per query): {prefix_ms:.1f} ms")
print(f"  Speedup: {naive_ms/prefix_ms:.0f}x")

Revenue days 100–200: $105,143
Direct sum (same answer):   $105,143

Why it matters — 10,000 random range queries:
  Naive  (O(n) per query): 33.6 ms
  Prefix (O(1) per query): 7.4 ms
  Speedup: 5x


### Under the hood: build once, query forever

```
arr    =  [ 3,  1,  4,  1,  5,  9]
index  =    0   1   2   3   4   5

prefix =  [ 0,  3,  4,  8,  9, 14, 23]
index  =    0   1   2   3   4   5   6
            ^
            sentinel: prefix[0] = 0

sum(arr[2:5]) = prefix[5] - prefix[2]
              = 14 - 4
              = 10  ✓  (4 + 1 + 5 = 10)
```
The sentinel `prefix[0] = 0` lets every query use the same formula, including queries that start at index 0.

---
### LC 303 — [Range Sum Query: Immutable](https://leetcode.com/problems/range-sum-query-immutable/)

> Implement `NumArray(nums)` with a method `sumRange(left, right)` that returns the sum of `nums[left..right]` inclusive. It will be called many times.
>
> Example: `NumArray([-2, 0, 3, -5, 2, -1]).sumRange(0, 2)` → `1`

**DS parallel:** Build a revenue lookup table once (`np.cumsum()`), answer thousands of ad-hoc date-range queries in O(1) each.

**Prefix sum insight:** Pay the O(n) build cost once in `__init__`. Every `sumRange` call is O(1) — one subtraction.

In [58]:
# ── NUMPY VERSION (what a DS would write) ─────────────────────────────────────
class NumArrayNumpy:
    def __init__(self, nums):
        self._prefix = np.concatenate([[0], np.cumsum(nums)])

    def sumRange(self, left, right):
        return int(self._prefix[right + 1] - self._prefix[left])

# ── RAW PYTHON (the interview answer) ─────────────────────────────────────────
class NumArray:
    def __init__(self, nums):
        self.prefix = [0] * (len(nums) + 1)
        for i, n in enumerate(nums):
            self.prefix[i + 1] = self.prefix[i] + n

    def sumRange(self, left, right):
        return self.prefix[right + 1] - self.prefix[left]

nums   = [-2, 0, 3, -5, 2, -1]
na_np  = NumArrayNumpy(nums)
na_raw = NumArray(nums)

queries = [(0, 2), (2, 5), (0, 5)]
print(f"{'query':<10} {'numpy':>8}  {'raw':>8}  {'direct sum':>10}")
print("-" * 44)
for l, r in queries:
    expected = sum(nums[l:r+1])
    np_ans   = na_np.sumRange(l, r)
    raw_ans  = na_raw.sumRange(l, r)
    match = "✓" if np_ans == raw_ans == expected else "✗"
    print(f"[{l},{r}]      {np_ans:>8}  {raw_ans:>8}  {expected:>10}  {match}")

query         numpy       raw  direct sum
--------------------------------------------
[0,2]             1         1           1  ✓
[2,5]            -1        -1          -1  ✓
[0,5]            -3        -3          -3  ✓


In [59]:
# ── Step-by-step trace — prefix construction + query visualization ─────────────
nums = [-2, 0, 3, -5, 2, -1]
print("Building the prefix array:")
print()
print(f"  {'step':>5}  {'nums[i]':>8}  {'prefix[i+1]':>12}  running formula")
print(f"  {'-----':>5}  {'-------':>8}  {'-----------':>12}  ---------------")

prefix = [0]
print(f"  {'0':>5}  {'(sentinel)':>8}  {0:>12}  prefix[0] = 0")
for i, n in enumerate(nums):
    new_val = prefix[-1] + n
    prefix.append(new_val)
    sign = '+' if n >= 0 else ''
    print(f"  {i+1:>5}  {n:>8}  {new_val:>12}  {prefix[-2]} {sign}{n} = {new_val}")

print()
print("Aligned view (offset by 1 — that's the point):")
print(f"  nums  :        " + ''.join(f"{v:>6}" for v in nums))
print(f"  index :        " + ''.join(f"{i:>6}" for i in range(len(nums))))
print(f"  prefix: " + ''.join(f"{v:>6}" for v in prefix))
print(f"  p_idx : " + ''.join(f"{i:>6}" for i in range(len(prefix))))
print()

# Trace each query with a visual showing which slice is covered
for l, r in [(0, 2), (2, 5), (0, 5)]:
    ans = prefix[r + 1] - prefix[l]
    span = ' + '.join(str(nums[i]) for i in range(l, r+1))

    # Visual: show which prefix slots are used
    p_markers = list(' ' * len(prefix))
    if l < len(prefix):     p_markers[l]     = 'i'
    if r+1 < len(prefix):   p_markers[r + 1] = 'j'

    print(f"  sumRange({l}, {r}):")
    print(f"    nums  :        " + ''.join(f"{v:>6}" for v in nums))
    print(f"    prefix: " + ''.join(f"{v:>6}" for v in prefix))
    print(f"    p_idx : " + ''.join(f"{i:>6}" for i in range(len(prefix))))
    print(f"    ptrs  : " + ''.join(f"{p:>6}" for p in p_markers))
    print(f"    → prefix[{r+1}] - prefix[{l}] = {prefix[r+1]} - {prefix[l]} = {ans}")
    print(f"    → nums[{l}..{r}] = [{span}] = {sum(nums[l:r+1])}  ✓")
    print()

Building the prefix array:

   step   nums[i]   prefix[i+1]  running formula
  -----   -------   -----------  ---------------
      0  (sentinel)             0  prefix[0] = 0
      1        -2            -2  0 -2 = -2
      2         0            -2  -2 +0 = -2
      3         3             1  -2 +3 = 1
      4        -5            -4  1 -5 = -4
      5         2            -2  -4 +2 = -2
      6        -1            -3  -2 -1 = -3

Aligned view (offset by 1 — that's the point):
  nums  :            -2     0     3    -5     2    -1
  index :             0     1     2     3     4     5
  prefix:      0    -2    -2     1    -4    -2    -3
  p_idx :      0     1     2     3     4     5     6

  sumRange(0, 2):
    nums  :            -2     0     3    -5     2    -1
    prefix:      0    -2    -2     1    -4    -2    -3
    p_idx :      0     1     2     3     4     5     6
    ptrs  :      i                 j                  
    → prefix[3] - prefix[0] = 1 - 0 = 1
    → nums[0..2] = [-2

---
### LC 560 — [Subarray Sum Equals K](https://leetcode.com/problems/subarray-sum-equals-k/)

> Count the number of contiguous subarrays whose elements sum to exactly `k`.
>
> Example: `nums = [1, 1, 1], k = 2` → `2`  (subarrays at indices 0–1 and 1–2)

**DS parallel:** Count date ranges in a P&L series netting to exactly zero — or any target profit.

**Key insight:** `prefix[j] - prefix[i] == k` means `nums[i..j-1]` sums to `k`. For each `j`, count previous `prefix[i]` values equal to `prefix[j] - k`. A hash map accumulates those counts in one pass — O(n). `seen[0] = 1` is the base case: subarray starting at index 0 needs an empty prefix with sum 0.

In [60]:
# ── NAIVE O(n²) ───────────────────────────────────────────────────────────────
def subarraySum_naive(nums, k):
    count = 0
    for i in range(len(nums)):
        s = 0
        for j in range(i, len(nums)):
            s += nums[j]
            if s == k: count += 1
    return count

# ── PREFIX SUM + HASH MAP (the interview answer) ───────────────────────────────
# O(n) time, O(n) space.
def subarraySum(nums, k):
    count  = 0
    prefix = 0
    seen   = defaultdict(int)
    seen[0] = 1              # base case: empty prefix has sum 0
    for n in nums:
        prefix += n
        count  += seen[prefix - k]   # subarrays ending here that sum to k
        seen[prefix] += 1
    return count

cases = [
    ([1, 1, 1],  2, 2),
    ([1, 2, 3],  3, 2),
    ([1, -1, 1], 0, 2),  # negative numbers — sliding window can't handle this
    ([1],        0, 0),
]
print(f"{'nums':<18} {'k':>3}  {'naive':>7}  {'prefix':>7}  {'expected':>8}")
print("-" * 52)
for nums, k, expected in cases:
    n_ans = subarraySum_naive(nums, k)
    p_ans = subarraySum(nums, k)
    match = "✓" if n_ans == p_ans == expected else "✗"
    print(f"{str(nums):<18} {k:>3}  {n_ans:>7}  {p_ans:>7}  {expected:>8}  {match}")

nums                 k    naive   prefix  expected
----------------------------------------------------
[1, 1, 1]            2        2        2         2  ✓
[1, 2, 3]            3        2        2         2  ✓
[1, -1, 1]           0        2        2         2  ✓
[1]                  0        0        0         0  ✓


In [61]:
# ── Step-by-step trace — running prefix, hash map lookup, and seen evolving ───
nums, k = [1, 1, 1], 2
print(f"nums = {nums},  k = {k}")
print(f"Goal: find subarrays where prefix[j] - prefix[i] == {k}")
print(f"      i.e., for each j, look for prefix[i] == prefix[j] - {k} in seen")
print()
print(f"  seen starts as {{0: 1}}  ← base case: empty prefix (sum=0) seen once")
print()
print(f"  {'i':>3}  {'n':>4}  {'prefix':>7}  {'lookup (p-k)':>12}  {'found':>6}  {'count':>6}  seen after")
print("  " + "-" * 72)

prefix = 0
seen   = defaultdict(int)
seen[0] = 1
count  = 0

for i, n in enumerate(nums):
    prefix      += n
    looking_for  = prefix - k
    found        = seen[looking_for]
    count       += found
    seen[prefix] += 1
    seen_str     = "{" + ", ".join(f"{kk}:{vv}" for kk, vv in sorted(seen.items())) + "}"
    found_note   = f"+{found}" if found else " 0"
    print(f"  {i:>3}  {n:>4}  {prefix:>7}  {looking_for:>12}  {found_note:>6}  {count:>6}  {seen_str}")

print()
print(f"Answer: {count}")
print()

# Verify by listing the actual subarrays
print("All subarrays summing to k (brute-force check):")
for i in range(len(nums)):
    s = 0
    for j in range(i, len(nums)):
        s += nums[j]
        if s == k:
            print(f"  nums[{i}:{j+1}] = {nums[i:j+1]},  sum = {s}  ✓")

print()
print("DS application — count date ranges netting to $0 P&L:")
daily_pnl    = [100, -50, -50, 200, -150, 50, -100]
zero_windows = subarraySum(daily_pnl, 0)
print(f"  Daily P&L:  {daily_pnl}")
print(f"  Windows netting to $0:  {zero_windows}")

nums = [1, 1, 1],  k = 2
Goal: find subarrays where prefix[j] - prefix[i] == 2
      i.e., for each j, look for prefix[i] == prefix[j] - 2 in seen

  seen starts as {0: 1}  ← base case: empty prefix (sum=0) seen once

    i     n   prefix  lookup (p-k)   found   count  seen after
  ------------------------------------------------------------------------
    0     1        1            -1       0       0  {-1:0, 0:1, 1:1}
    1     1        2             0      +1       1  {-1:0, 0:1, 1:1, 2:1}
    2     1        3             1      +1       2  {-1:0, 0:1, 1:1, 2:1, 3:1}

Answer: 2

All subarrays summing to k (brute-force check):
  nums[0:2] = [1, 1],  sum = 2  ✓
  nums[1:3] = [1, 1],  sum = 2  ✓

DS application — count date ranges netting to $0 P&L:
  Daily P&L:  [100, -50, -50, 200, -150, 50, -100]
  Windows netting to $0:  5


---
## Summary

| Pattern | LeetCode | DS/ML Equivalent | Time | Space |
|---------|----------|------------------|------|-------|
| Two pointers (opposite ends) | LC 167 Two Sum II | `np.searchsorted` on sorted pairs | O(n) | O(1) |
| Two pointers (skip non-alnum) | LC 125 Valid Palindrome | `str[::-1]` comparison | O(n) | O(1) |
| Sliding window (fixed k) | LC 643 Max Avg Subarray | `pd.rolling(k).mean()` | O(n) | O(1) |
| Sliding window (variable) | LC 3 Longest No-Repeat | No pandas equivalent — write it yourself | O(n) | O(k) |
| Prefix sum | LC 303 Range Sum Query | `np.cumsum()` | O(n) build, O(1) query | O(n) |
| Prefix sum + hash map | LC 560 Subarray Sum = K | Balanced window counting | O(n) | O(n) |

**The mental model shift:** pandas/numpy trains you to think in whole-array operations — you hand the data to a function and get a result. These patterns require the opposite: you're an agent *inside* the array, moving step by step, maintaining a small amount of local state (`window_sum`, `freq`, `prefix`). The interview tests whether you can operate in that mode. Both modes are valuable; switching deliberately between them is the skill.